# Navigation mini-project on Google Colab

**Before you start**
1. On your own computer, clone the repository (https://github.com/UCLAdeepvision/CS163-Projects-2026Fall.git) and save `nav-assets.tar` (the model and the scenes) and `processed-data.tar` (the training data) in the `nav-project` folder, and upload the whole `nav-project` folder to your Google Drive (My Drive), so that you have `My Drive/nav-project` with the two tar files in it. Open this notebook from there (right-click it in Drive → Open with → Google Colaboratory).
2. Runtime → Change runtime type → **T4 GPU** (a different NVIDIA GPU could also work).
3. Run the cells from top to bottom. They install the packages, unpack the model, the scenes and the training data (`unpack_files.py`), and check the setup. `data.tar` (the raw clips) is optional and is not used here.

**Storage.** Colab erases its own disk when a session ends, but not your Drive. So the code (including your edited
`train.py`) and everything you produce stay in your Drive folder, and only the large training data is copied to Colab's disk
each session. If a session ends, start again from the top: the cells skip what is already done.

Edit `code/train.py` in the Colab file browser (left side: `drive/MyDrive/nav-project/mini-project-nav/code/train.py`).

## 1. GPU, Drive and the code

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/nav-project'   # the folder you uploaded

In [ ]:
PKG = f'{DRIVE}/mini-project-nav'
assert os.path.exists(PKG), f'{PKG} is missing: upload the nav-project folder (the repository you cloned) to your Google Drive first'
%cd {PKG}

## 2. Install

In [ ]:
!pip install -q -r requirements.txt     # torch is already installed on Colab and is left alone

## 3. Unpack the model, the scenes and the data
The model and the scenes are unpacked into your Drive folder the first time (a minute or two) and skipped afterwards. The training data is unpacked to Colab's disk every session, from the `processed-data.tar` in your Drive folder.

In [ ]:
DATA = '/content/data/processed'                  # local disk: fast, but erased when the session ends
!python unpack_files.py --data {DATA}
!ls -lh {DATA}

## 4. Check the setup

In [ ]:
!python check_setup.py

## 5. Train

`train.py` first prints the validation error of the original model and of a trivial baseline, then stops at your first TODO.
Run it once before you change anything. Checkpoints go straight to Drive, so a disconnect does not lose them.
`--workers 2` because Colab has only two CPU cores.

In [ ]:
RUN = f'{DRIVE}/runs/first'
!python code/train.py --data {DATA} --out {RUN} --epochs 4 --workers 2

## 6. Export to ONNX and check the file

In [ ]:
import sys; sys.path.append("code")
from vint import Waypoints, load_model
from export_onnx import export_module
export_module(Waypoints(load_model(f"{RUN}/best.pt")), f"{DRIVE}/submission.onnx")
!python check_submission.py {DRIVE}/submission.onnx

## 7. Score it in the simulator
Runs the 53 development episodes with your ONNX file (no window; the numbers are what counts). Use `--scenes <name>` to try one scene first.

In [ ]:
!OMP_NUM_THREADS=2 python evaluate.py --policy code/onnx_policy.py --onnx {DRIVE}/submission.onnx --out {DRIVE}/results.json

## 8. Watch it drive (on your own computer)

Colab cannot show the viewer, so this part runs on your own machine, with the setup from the handout.

1. Download `submission.onnx` from your Drive folder `nav-project` into the `mini-project-nav` folder.
2. Start the viewer:

```
python viewer.py --policy code/onnx_policy.py --onnx submission.onnx
```

3. Open http://localhost:8080 in a browser. Keep that tab visible: the browser draws the robot's camera pictures. They differ slightly from the ones `evaluate.py` renders, so the scores of `evaluate.py` are the ones that count.